# Health Insurance Lead Prediction: Data Cleaning & Exploratory Data Analysis (EDA)

In this notebook, we will load the raw dataset, handle missing values, and perform exploratory data analysis to understand the key factors driving health insurance lead conversions.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Create visualizations directory if it doesn't exist
os.makedirs('../visualizations', exist_ok=True)

# Load the dataset
# We use openpyxl engine for .xlsx files
try:
    df = pd.read_excel('../data/raw/Health Insurance Lead Prediction.xlsx', sheet_name='Sheet1')
    print(f"Dataset loaded successfully with {df.shape[0]} rows and {df.shape[1]} columns.")
except Exception as e:
    print(f"Error loading dataset: {e}")


In [ ]:
# Display the first few rows
df.head()


In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
print("Missing Values:\n", missing_values[missing_values > 0])


In [ ]:
# Handle missing values
# Drop rows where target variable is missing or invalid
if 'Response' in df.columns:
    df = df.dropna(subset=['Response'])
    df = df[df['Response'].isin([0, 1])]

# Impute Health Indicator with mode
if 'Health Indicator' in df.columns:
    df['Health Indicator'] = df['Health Indicator'].fillna(df['Health Indicator'].mode()[0])

# Impute Holding_Policy_Duration with median or mode, or treat as a separate category
if 'Holding_Policy_Duration' in df.columns:
    # Since it's categorical (e.g., '14+', '1.0'), we replace missing with '0' (No previous policy)
    df['Holding_Policy_Duration'] = df['Holding_Policy_Duration'].fillna('0')
    
# Verify missing values are handled
print("Missing values after cleaning:\n", df.isnull().sum().sum())


In [ ]:
# Exploratory Data Analysis (EDA)
sns.set_style('whitegrid')

# 1. Lead Conversion Rate
plt.figure(figsize=(6, 4))
ax = sns.countplot(x='Response', hue='Response', data=df, palette='viridis', legend=False)
plt.title('Distribution of Lead Conversion (Response)')
plt.xlabel('Response (0 = No, 1 = Yes)')
plt.ylabel('Count')

# Annotate bars
for p in ax.patches:
    ax.annotate(f'{p.get_height()}', (p.get_x() + p.get_width() / 2., p.get_height()),
                ha='center', va='bottom')

plt.tight_layout()
plt.savefig('../visualizations/lead_conversion_rate.png', dpi=300)
plt.show()


In [ ]:
# 2. Conversion by Accomodation Type
plt.figure(figsize=(8, 5))
sns.countplot(x='Accomodation_Type', hue='Response', data=df, palette='Set2')
plt.title('Lead Conversion by Accommodation Type')
plt.xlabel('Accommodation Type')
plt.ylabel('Count')
plt.legend(title='Response')
plt.tight_layout()
plt.savefig('../visualizations/conversion_by_accommodation.png', dpi=300)
plt.show()


In [ ]:
# 3. Age Distribution of Leads vs Non-Leads
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Upper_Age', hue='Response', bins=30, kde=True, palette='coolwarm')
plt.title('Age Distribution by Lead Conversion')
plt.xlabel('Upper Age')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../visualizations/age_distribution.png', dpi=300)
plt.show()


In [ ]:
# 4. Impact of Recommended Policy Premium
plt.figure(figsize=(10, 6))
sns.boxplot(x='Response', y='Reco_Policy_Premium', hue='Response', data=df, palette='pastel', legend=False)
plt.title('Recommended Policy Premium vs. Lead Conversion')
plt.xlabel('Response')
plt.ylabel('Recommended Premium')
plt.tight_layout()
plt.savefig('../visualizations/premium_vs_conversion.png', dpi=300)
plt.show()


In [ ]:
# Save the cleaned dataset for the modeling phase
df.to_csv('../data/processed/cleaned_health_insurance.csv', index=False)
print("Cleaned dataset saved to data/processed/")
